# LaBSE Native Indic–Indic All-Pairs Fine-tuning

This notebook trains **one LaBSE model** on **all 462 directed Indic–Indic language pairs** from IN22-Gen.

It is the next step after the 13-model diagnostic experiment.

## What this notebook does

- Uses `sentence-transformers/LaBSE` as the base model.
- Uses **only Indic–Indic pairs**, not English pivot pairs.
- Uses all 22 scheduled Indic languages from IN22.
- Generates all directed pairs: `22 × 21 = 462`.
- Trains one balanced all-pairs model.
- Saves only:
  - `best_model/`
  - `final_model/`
  - `train_metrics.csv`
  - `training_config.json`
  - train/validation pair CSVs
  - IN22-Conv evaluation CSVs

No rolling checkpoints are saved.

In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
# Install dependencies
!pip -q install -U "sentence-transformers" "datasets" "accelerate" "transformers>=4.51.0,<5"     "huggingface_hub" "tqdm" "pandas==2.2.2" "numpy==2.0.2" "scikit-learn>=1.5,<1.9"

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import random
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# -------------------------
# Project directory
# -------------------------
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = Path("/content/drive/MyDrive/labse_all_pairs_indic_finetuning")
else:
    PROJECT_DIR = Path("/content/labse_all_pairs_indic_finetuning")

OUTPUT_DIR = PROJECT_DIR / "outputs"
METRICS_DIR = PROJECT_DIR / "metrics"
DATA_DIR = PROJECT_DIR / "data"

for d in [OUTPUT_DIR, METRICS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_NAME = "labse_all_462_directed_pairs_balanced"
RUN_DIR = OUTPUT_DIR / RUN_NAME
BEST_MODEL_DIR = RUN_DIR / "best_model"
FINAL_MODEL_DIR = RUN_DIR / "final_model"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Run dir:", RUN_DIR)
print("Best model dir:", BEST_MODEL_DIR)
print("Final model dir:", FINAL_MODEL_DIR)

In [ ]:
# -------------------------
# Hyperparameters
# -------------------------
BASE_MODEL_NAME = "sentence-transformers/LaBSE"
MAX_SEQ_LENGTH = 128
BATCH_SIZE = 32

# Conservative all-pair fine-tuning:
# 462 pairs × 100 examples = 46,200 pairs before validation split.
# Increase to 250/500/1000 only after this small run is stable.
EXAMPLES_PER_DIRECTED_PAIR = 100

EPOCHS = 2
LEARNING_RATE = 2e-6
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
VAL_SIZE = 0.10
EVAL_BATCH_SIZE = 128
USE_AMP = torch.cuda.is_available()

# Model-level resume only. This is not checkpointing.
SKIP_IF_COMPLETE = True
BEST_MODEL_METRIC = "val_cosine_gap"

# Final evaluation on IN22-Conv. 0 means all rows. Use 300 for a quick dry run.
EVAL_MAX_ROWS = 0

print("Base model:", BASE_MODEL_NAME)
print("Examples per directed pair:", EXAMPLES_PER_DIRECTED_PAIR)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Validation size:", VAL_SIZE)
print("USE_AMP:", USE_AMP)

In [ ]:
# -------------------------
# IN22 language map
# -------------------------
# English is listed only for column reference. Training explicitly excludes it.
LANG_CODE = {
    "asm": "asm_Beng", "ben": "ben_Beng", "brx": "brx_Deva", "doi": "doi_Deva",
    "guj": "guj_Gujr", "hin": "hin_Deva", "kan": "kan_Knda", "kas": "kas_Arab",
    "gom": "gom_Deva", "mai": "mai_Deva", "mal": "mal_Mlym", "mni": "mni_Mtei",
    "mar": "mar_Deva", "npi": "npi_Deva", "ory": "ory_Orya", "pan": "pan_Guru",
    "san": "san_Deva", "sat": "sat_Olck", "snd": "snd_Deva", "tam": "tam_Taml",
    "tel": "tel_Telu", "urd": "urd_Arab", "eng": "eng_Latn",
}

INDIC_LANGS = [lang for lang in LANG_CODE if lang != "eng"]
ALL_DIRECTED_PAIRS = [(src, tgt) for src in INDIC_LANGS for tgt in INDIC_LANGS if src != tgt]

print("Number of Indic languages:", len(INDIC_LANGS))
print("Number of directed Indic-Indic pairs:", len(ALL_DIRECTED_PAIRS))
print("First 10 pairs:", ALL_DIRECTED_PAIRS[:10])

assert len(INDIC_LANGS) == 22
assert len(ALL_DIRECTED_PAIRS) == 462
assert all(src != "eng" and tgt != "eng" for src, tgt in ALL_DIRECTED_PAIRS)
print("Verified: 462 directed pairs, no English pivot.")

In [ ]:
# -------------------------
# Load IN22-Gen for training
# -------------------------
IN22_GEN_NAME = "ai4bharat/IN22-Gen"
in22_gen = load_dataset(IN22_GEN_NAME, "default", split="test")
in22_gen_df = in22_gen.to_pandas()

print("IN22-Gen shape:", in22_gen_df.shape)
print("Columns:", list(in22_gen_df.columns)[:10], "...")
display(in22_gen_df.head(2))

In [ ]:
def resolve_sentence_col(short_lang, df):
    code = LANG_CODE[short_lang]
    candidates = [code, f"sentence_{code}", short_lang, f"sentence_{short_lang}"]
    if short_lang == "snd":
        candidates += ["snd_Arab", "sentence_snd_Arab"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Could not find sentence column for {short_lang}. Tried {candidates}")

SENTENCE_COLS = {lang: resolve_sentence_col(lang, in22_gen_df) for lang in INDIC_LANGS}
print("Resolved sentence columns:")
for lang, col in SENTENCE_COLS.items():
    print(f"{lang:>3} -> {col}")

In [ ]:
# -------------------------
# Build balanced all-pairs train/validation data
# -------------------------
def clean_text_series(s):
    return s.astype(str).str.strip()


def build_all_pairs_dataframe(df, directed_pairs, sentence_cols, examples_per_pair, seed=42):
    rng = np.random.default_rng(seed)
    max_rows = len(df)
    n = min(examples_per_pair, max_rows)

    # Same row IDs for all directions, so every pair gets equal semantic coverage.
    selected_row_ids = np.sort(rng.choice(max_rows, size=n, replace=False))

    pieces = []
    for src, tgt in tqdm(directed_pairs, desc="Generating directed pair dataframe"):
        src_col = sentence_cols[src]
        tgt_col = sentence_cols[tgt]
        tmp = pd.DataFrame({
            "row_id": selected_row_ids,
            "src_lang": src,
            "tgt_lang": tgt,
            "direction": f"{src}->{tgt}",
            "sentence1": clean_text_series(df.iloc[selected_row_ids][src_col]).values,
            "sentence2": clean_text_series(df.iloc[selected_row_ids][tgt_col]).values,
        })
        tmp = tmp[
            (tmp["sentence1"].notna()) &
            (tmp["sentence2"].notna()) &
            (tmp["sentence1"].str.len() > 0) &
            (tmp["sentence2"].str.len() > 0)
        ]
        pieces.append(tmp)
    return pd.concat(pieces, ignore_index=True)


def row_id_train_val_split(pair_df, val_size=0.10, seed=42):
    # Split by row_id, not individual rows, to avoid same semantic row in train and val.
    rng = np.random.default_rng(seed)
    row_ids = np.array(sorted(pair_df["row_id"].unique()))
    rng.shuffle(row_ids)
    n_val = max(1, int(round(len(row_ids) * val_size)))
    val_row_ids = set(row_ids[:n_val])
    train_row_ids = set(row_ids[n_val:])
    train_df = pair_df[pair_df["row_id"].isin(train_row_ids)].reset_index(drop=True)
    val_df = pair_df[pair_df["row_id"].isin(val_row_ids)].reset_index(drop=True)
    return train_df, val_df

all_pairs_df = build_all_pairs_dataframe(
    df=in22_gen_df,
    directed_pairs=ALL_DIRECTED_PAIRS,
    sentence_cols=SENTENCE_COLS,
    examples_per_pair=EXAMPLES_PER_DIRECTED_PAIR,
    seed=SEED,
)
train_df, val_df = row_id_train_val_split(all_pairs_df, val_size=VAL_SIZE, seed=SEED)

print("Total pair rows:", len(all_pairs_df))
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Unique directions:", all_pairs_df["direction"].nunique())
print("Train row_ids:", train_df["row_id"].nunique())
print("Val row_ids:", val_df["row_id"].nunique())

assert all_pairs_df["direction"].nunique() == 462
assert train_df["direction"].nunique() == 462
assert val_df["direction"].nunique() == 462

all_pairs_df.to_csv(DATA_DIR / "all_462_directed_pairs_all_rows.csv", index=False)
train_df.to_csv(DATA_DIR / "train_pairs_used.csv", index=False)
val_df.to_csv(DATA_DIR / "val_pairs_used.csv", index=False)

display(all_pairs_df.head())
display(train_df.groupby("direction").size().describe())

In [ ]:
# -------------------------
# Dataset, DataLoader, and losses
# -------------------------
class PairTextDataset(Dataset):
    def __init__(self, df):
        self.sentence1 = df["sentence1"].astype(str).tolist()
        self.sentence2 = df["sentence2"].astype(str).tolist()
    def __len__(self):
        return len(self.sentence1)
    def __getitem__(self, idx):
        return self.sentence1[idx], self.sentence2[idx]


def batch_to_device(sentence_features, device):
    moved = []
    for feature_dict in sentence_features:
        moved.append({k: v.to(device) if torch.is_tensor(v) else v for k, v in feature_dict.items()})
    return moved


def make_dataloader(model, df, batch_size):
    dataset = PairTextDataset(df)
    def collate_fn(batch):
        texts1 = [x[0] for x in batch]
        texts2 = [x[1] for x in batch]
        sentence_features = [model.tokenize(texts1), model.tokenize(texts2)]
        labels = torch.arange(len(batch), dtype=torch.long)
        return sentence_features, labels
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True, collate_fn=collate_fn, num_workers=0)


def symmetric_mnrl_loss(src_emb, tgt_emb, scale=20.0):
    src_emb = F.normalize(src_emb, p=2, dim=1)
    tgt_emb = F.normalize(tgt_emb, p=2, dim=1)
    scores = torch.matmul(src_emb, tgt_emb.T) * scale
    labels = torch.arange(scores.size(0), device=scores.device)
    return (F.cross_entropy(scores, labels) + F.cross_entropy(scores.T, labels)) / 2.0


@torch.no_grad()
def compute_validation_metrics(model, val_df, batch_size=128):
    model.eval()
    records = []
    for direction, g in tqdm(val_df.groupby("direction"), desc="Validation by direction", leave=False):
        if len(g) < 2:
            continue
        src_texts = g["sentence1"].astype(str).tolist()
        tgt_texts = g["sentence2"].astype(str).tolist()
        src_emb = model.encode(src_texts, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        tgt_emb = model.encode(tgt_texts, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        gold = np.sum(src_emb * tgt_emb, axis=1)
        random_neg = np.sum(src_emb * np.roll(tgt_emb, shift=1, axis=0), axis=1)
        threshold = (float(gold.mean()) + float(random_neg.mean())) / 2.0
        sensitivity = float((gold >= threshold).mean())
        specificity = float((random_neg < threshold).mean())
        records.append({
            "direction": direction,
            "n": len(g),
            "mean_gold_cosine": float(gold.mean()),
            "mean_random_cosine": float(random_neg.mean()),
            "cosine_gap": float(gold.mean() - random_neg.mean()),
            "sensitivity_midpoint": sensitivity,
            "specificity_midpoint": specificity,
            "balanced_accuracy_midpoint": (sensitivity + specificity) / 2.0,
        })
    per_direction = pd.DataFrame(records)
    summary = {
        "val_mean_gold_cosine": float(np.average(per_direction["mean_gold_cosine"], weights=per_direction["n"])),
        "val_mean_random_cosine": float(np.average(per_direction["mean_random_cosine"], weights=per_direction["n"])),
        "val_cosine_gap": float(np.average(per_direction["cosine_gap"], weights=per_direction["n"])),
        "val_sensitivity_midpoint": float(np.average(per_direction["sensitivity_midpoint"], weights=per_direction["n"])),
        "val_specificity_midpoint": float(np.average(per_direction["specificity_midpoint"], weights=per_direction["n"])),
        "val_balanced_accuracy_midpoint": float(np.average(per_direction["balanced_accuracy_midpoint"], weights=per_direction["n"])),
    }
    return summary, per_direction

In [ ]:
# -------------------------
# Training loop
# -------------------------
def train_all_pairs_model():
    config = {
        "run_name": RUN_NAME,
        "base_model": BASE_MODEL_NAME,
        "training_type": "balanced_all_462_directed_indic_indic_pairs",
        "num_indic_languages": len(INDIC_LANGS),
        "num_directed_pairs": len(ALL_DIRECTED_PAIRS),
        "examples_per_directed_pair": EXAMPLES_PER_DIRECTED_PAIR,
        "total_pair_rows_before_split": int(len(all_pairs_df)),
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "val_size": VAL_SIZE,
        "max_seq_length": MAX_SEQ_LENGTH,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "max_grad_norm": MAX_GRAD_NORM,
        "best_model_metric": BEST_MODEL_METRIC,
        "uses_english_pivot": False,
        "languages": INDIC_LANGS,
        "directed_pairs": [f"{s}->{t}" for s, t in ALL_DIRECTED_PAIRS],
    }
    with open(RUN_DIR / "training_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, ensure_ascii=False)

    if SKIP_IF_COMPLETE and (BEST_MODEL_DIR / "modules.json").exists() and (FINAL_MODEL_DIR / "modules.json").exists():
        print("Both best_model and final_model already exist. Skipping training.")
        return pd.read_csv(RUN_DIR / "train_metrics.csv") if (RUN_DIR / "train_metrics.csv").exists() else pd.DataFrame()

    model = SentenceTransformer(BASE_MODEL_NAME, device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH
    model.train()
    train_loader = make_dataloader(model, train_df, BATCH_SIZE)

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 1.0 - progress)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    print("Training rows:", len(train_df))
    print("Batches per epoch:", len(train_loader))
    print("Total steps:", total_steps)
    print("Warmup steps:", warmup_steps)

    metrics_rows = []
    best_score = -1e9
    global_step = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_losses = []
        pbar = tqdm(train_loader, desc=f"{RUN_NAME} | epoch {epoch}/{EPOCHS}")
        for sentence_features, labels in pbar:
            sentence_features = batch_to_device(sentence_features, DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                src_emb = model(sentence_features[0])["sentence_embedding"]
                tgt_emb = model(sentence_features[1])["sentence_embedding"]
                loss = symmetric_mnrl_loss(src_emb, tgt_emb, scale=20.0)
            scaler.scale(loss).backward()
            if MAX_GRAD_NORM is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            global_step += 1
            loss_float = float(loss.detach().cpu().item())
            epoch_losses.append(loss_float)
            pbar.set_postfix({"loss": f"{loss_float:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

        mean_train_loss = float(np.mean(epoch_losses))
        val_summary, val_per_direction = compute_validation_metrics(model, val_df, batch_size=EVAL_BATCH_SIZE)
        val_per_direction.to_csv(RUN_DIR / f"val_per_direction_epoch_{epoch}.csv", index=False)
        row = {
            "run_name": RUN_NAME,
            "epoch": epoch,
            "global_step": global_step,
            "mean_train_loss": mean_train_loss,
            "learning_rate": float(scheduler.get_last_lr()[0]),
            **val_summary,
        }
        metrics_rows.append(row)
        pd.DataFrame(metrics_rows).to_csv(RUN_DIR / "train_metrics.csv", index=False)
        current_score = row[BEST_MODEL_METRIC]
        print(f"Epoch {epoch} summary:")
        print(json.dumps(row, indent=2))
        if current_score > best_score:
            best_score = current_score
            if BEST_MODEL_DIR.exists():
                shutil.rmtree(BEST_MODEL_DIR)
            model.save(str(BEST_MODEL_DIR))
            print(f"Saved new best_model with {BEST_MODEL_METRIC}={best_score:.6f}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if FINAL_MODEL_DIR.exists():
        shutil.rmtree(FINAL_MODEL_DIR)
    model.save(str(FINAL_MODEL_DIR))
    print("Saved final_model:", FINAL_MODEL_DIR)

    verification = {
        "run_name": RUN_NAME,
        "best_model_exists": (BEST_MODEL_DIR / "modules.json").exists(),
        "final_model_exists": (FINAL_MODEL_DIR / "modules.json").exists(),
        "train_metrics_exists": (RUN_DIR / "train_metrics.csv").exists(),
        "training_config_exists": (RUN_DIR / "training_config.json").exists(),
        "num_directed_pairs": len(ALL_DIRECTED_PAIRS),
        "uses_english_pivot": False,
    }
    pd.DataFrame([verification]).to_csv(RUN_DIR / "model_generation_verification.csv", index=False)
    print("Verification:")
    print(json.dumps(verification, indent=2))
    assert verification["best_model_exists"]
    assert verification["final_model_exists"]
    assert verification["num_directed_pairs"] == 462
    assert verification["uses_english_pivot"] is False
    return pd.DataFrame(metrics_rows)

train_metrics_df = train_all_pairs_model()
display(train_metrics_df)

In [ ]:
# -------------------------
# Load IN22-Conv for final evaluation
# -------------------------
IN22_CONV_NAME = "ai4bharat/IN22-Conv"
in22_conv = load_dataset(IN22_CONV_NAME, "default", split="test")
in22_conv_df = in22_conv.to_pandas()
if EVAL_MAX_ROWS and EVAL_MAX_ROWS > 0:
    in22_conv_df = in22_conv_df.head(EVAL_MAX_ROWS).copy()
print("IN22-Conv shape used for eval:", in22_conv_df.shape)
display(in22_conv_df.head(2))

CONV_SENTENCE_COLS = {lang: resolve_sentence_col(lang, in22_conv_df) for lang in INDIC_LANGS}
print("Resolved IN22-Conv columns.")

In [ ]:
# -------------------------
# Final evaluation: baseline vs best vs final
# -------------------------
@torch.no_grad()
def encode_language_cache(model, df, sentence_cols, langs, batch_size=128):
    cache = {}
    for lang in tqdm(langs, desc="Encoding languages"):
        col = sentence_cols[lang]
        texts = df[col].astype(str).str.strip().tolist()
        emb = model.encode(texts, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        cache[lang] = emb
    return cache


def retrieval_metrics_for_pair(src_emb, tgt_emb):
    sim = np.matmul(src_emb, tgt_emb.T)
    n = sim.shape[0]
    correct = np.arange(n)
    top1 = np.argmax(sim, axis=1)
    correct_scores = sim[correct, correct]
    ranks = 1 + np.sum(sim > correct_scores[:, None], axis=1)
    return {
        "accuracy_at_1": float(np.mean(top1 == correct)),
        "recall_at_5": float(np.mean(ranks <= 5)),
        "recall_at_10": float(np.mean(ranks <= 10)),
        "mrr": float(np.mean(1.0 / ranks)),
    }


def separation_metrics_for_pair(src_emb, tgt_emb):
    gold = np.sum(src_emb * tgt_emb, axis=1)
    random_neg = np.sum(src_emb * np.roll(tgt_emb, shift=1, axis=0), axis=1)
    mean_gold = float(gold.mean())
    mean_random = float(random_neg.mean())
    threshold = (mean_gold + mean_random) / 2.0
    sensitivity = float((gold >= threshold).mean())
    specificity = float((random_neg < threshold).mean())
    return {
        "mean_gold_cosine": mean_gold,
        "std_gold_cosine": float(gold.std()),
        "mean_random_cosine": mean_random,
        "std_random_cosine": float(random_neg.std()),
        "cosine_gap": mean_gold - mean_random,
        "sensitivity_midpoint": sensitivity,
        "specificity_midpoint": specificity,
        "balanced_accuracy_midpoint": (sensitivity + specificity) / 2.0,
    }


def evaluate_model_on_all_pairs(model_name, model_path_or_name):
    print(f"Loading model: {model_name}")
    model = SentenceTransformer(str(model_path_or_name), device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH
    model.eval()
    emb_cache = encode_language_cache(model, in22_conv_df, CONV_SENTENCE_COLS, INDIC_LANGS, batch_size=EVAL_BATCH_SIZE)
    rows = []
    for src, tgt in tqdm(ALL_DIRECTED_PAIRS, desc=f"Evaluating {model_name}"):
        src_emb = emb_cache[src]
        tgt_emb = emb_cache[tgt]
        rows.append({
            "model": model_name,
            "src_lang": src,
            "tgt_lang": tgt,
            "direction": f"{src}->{tgt}",
            "n": len(in22_conv_df),
            **retrieval_metrics_for_pair(src_emb, tgt_emb),
            **separation_metrics_for_pair(src_emb, tgt_emb),
        })
    out = pd.DataFrame(rows)
    del model, emb_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

models_to_eval = {
    "labse_baseline": BASE_MODEL_NAME,
    "all_462_best_model": BEST_MODEL_DIR,
    "all_462_final_model": FINAL_MODEL_DIR,
}

all_eval_dfs = []
for model_name, model_path in models_to_eval.items():
    if model_name != "labse_baseline" and not (Path(model_path) / "modules.json").exists():
        print(f"Skipping {model_name}; model folder missing:", model_path)
        continue
    df_eval = evaluate_model_on_all_pairs(model_name, model_path)
    df_eval.to_csv(METRICS_DIR / f"in22_conv_eval_{model_name}_by_direction.csv", index=False)
    all_eval_dfs.append(df_eval)

eval_all_df = pd.concat(all_eval_dfs, ignore_index=True)
eval_all_df.to_csv(METRICS_DIR / "in22_conv_eval_all_models_by_direction.csv", index=False)
display(eval_all_df.head())
print("Saved:", METRICS_DIR / "in22_conv_eval_all_models_by_direction.csv")

In [ ]:
# -------------------------
# Evaluation summary and delta vs baseline
# -------------------------
summary_cols = [
    "accuracy_at_1", "recall_at_5", "recall_at_10", "mrr",
    "mean_gold_cosine", "mean_random_cosine", "cosine_gap",
    "sensitivity_midpoint", "specificity_midpoint", "balanced_accuracy_midpoint",
]

summary_df = eval_all_df.groupby("model")[summary_cols].mean().reset_index().sort_values("cosine_gap", ascending=False)
summary_df.to_csv(METRICS_DIR / "in22_conv_eval_summary_by_model.csv", index=False)
display(summary_df)

baseline_row = summary_df[summary_df["model"] == "labse_baseline"].iloc[0]
delta_rows = []
for _, row in summary_df.iterrows():
    if row["model"] == "labse_baseline":
        continue
    d = {"model": row["model"]}
    for col in summary_cols:
        d[f"delta_{col}"] = float(row[col] - baseline_row[col])
    delta_rows.append(d)

delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(METRICS_DIR / "delta_vs_labse_baseline.csv", index=False)
display(delta_df)

In [ ]:
# -------------------------
# Pair-level improvement/degradation tables
# -------------------------
baseline = eval_all_df[eval_all_df["model"] == "labse_baseline"].copy()

for model_name in [m for m in eval_all_df["model"].unique() if m != "labse_baseline"]:
    model_df = eval_all_df[eval_all_df["model"] == model_name].copy()
    merged = model_df.merge(baseline, on=["src_lang", "tgt_lang", "direction"], suffixes=("", "_baseline"))
    for col in summary_cols:
        merged[f"delta_{col}"] = merged[col] - merged[f"{col}_baseline"]
    delta_pair_path = METRICS_DIR / f"delta_{model_name}_vs_labse_baseline_by_direction.csv"
    merged.to_csv(delta_pair_path, index=False)

    print("
Model:", model_name)
    print("Top 10 improved directions by delta cosine_gap")
    display(merged[["direction", "delta_cosine_gap", "delta_accuracy_at_1", "delta_specificity_midpoint"]].sort_values("delta_cosine_gap", ascending=False).head(10))

    print("Bottom 10 degraded directions by delta cosine_gap")
    display(merged[["direction", "delta_cosine_gap", "delta_accuracy_at_1", "delta_specificity_midpoint"]].sort_values("delta_cosine_gap", ascending=True).head(10))

print("Saved pair-level delta files in:", METRICS_DIR)

In [ ]:
# -------------------------
# Zip metrics only
# -------------------------
# Models are already saved in Drive under RUN_DIR.
# This zip is small and only contains CSV/JSON/TXT reports, not model weights.
import zipfile

EXPORT_DIR = PROJECT_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
zip_path = EXPORT_DIR / "labse_all_pairs_metrics_only.zip"
keep_suffixes = {".csv", ".json", ".txt"}

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in PROJECT_DIR.rglob("*"):
        if not file.is_file():
            continue
        path_str = str(file)
        if "/best_model/" in path_str or "/final_model/" in path_str:
            continue
        if "/exports/" in path_str and file.suffix == ".zip":
            continue
        if file.suffix.lower() in keep_suffixes:
            z.write(file, arcname=file.relative_to(PROJECT_DIR))

print("Saved metrics zip:", zip_path)
print("Zip size MB:", round(zip_path.stat().st_size / (1024 ** 2), 2))

## How to interpret the result

The all-pairs model is successful only if it improves weak directions **without collapsing specificity**.

Check:

- `delta_accuracy_at_1`
- `delta_cosine_gap`
- `delta_specificity_midpoint`
- pair-level improvement/degradation tables

A bad model may increase sensitivity but reduce specificity. That means it is scoring too many wrong pairs as similar.

The expected research conclusion from this notebook should be one of:

1. **Balanced all-pair fine-tuning improves global alignment**.
2. **Balanced all-pair fine-tuning improves weak pairs but hurts strong pairs**.
3. **Balanced all-pair fine-tuning is unstable**, meaning the next experiment should use weak-pair weighting and/or preservation loss.